# 13 — Qwen Kidney-Transplant Baseline: Test and Validation with Unsloth

**Environment:** Google Colab with one NVIDIA CUDA GPU  
**Model:** the completed kidney-trained Qwen model saved by the preceding Unsloth `SFTTrainer` notebook  
**Purpose:** verify that the saved model reloads correctly, test it in inference mode, evaluate the held-out acute-rejection task, and measure the auxiliary recipient factual-recall task.

> **This notebook does not train or unlearn the model.** It treats the completed model as frozen.

The preceding training run contained **49,028 SFT examples**. The frozen project split contains **42,024 training assessments** and **7,004 training recipients**, which explains that total as:

`42,024 clinical-assessment examples + 7,004 recipient factual-recall examples = 49,028 SFT examples`.

The notebook keeps these two tasks separate during evaluation so that classification utility and factual recall are not mixed into one score.

## Notebook pipeline

1. Verify the Colab CUDA environment.
2. Install/import Unsloth and the evaluation libraries.
3. Restore the saved `final_model` produced by the training notebook.
4. Reload the model using Unsloth and explicitly switch it to inference mode.
5. Load the frozen kidney-transplant dataset, feature contract and train/validation/test memberships.
6. Verify the expected **42,024 + 7,004 = 49,028** training-example structure.
7. Recover and verify the exact prompt contract used during SFT.
8. Run a small deterministic inference smoke test before any full evaluation.
9. Evaluate the **validation split** and select the classification threshold using validation data only.
10. Evaluate the **test split once** using the frozen threshold.
11. Evaluate factual recall on a deterministic sample of training recipients and an unseen-recipient control sample.
12. Save predictions, metrics, prompt-contract evidence and verification checks.
13. Package all evaluation outputs so they can be downloaded from Colab.

The notebook deliberately fails before the full metrics if the prompt contract has not been confirmed. A model can only be evaluated fairly with the same prompt/answer format it was trained on.

## 1. Unsloth is the source of truth for model loading and inference

This notebook follows the official Unsloth workflow rather than inventing a second model-loading path.

The relevant Unsloth documentation is:

- **Datasets Guide:** https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/datasets-guide
- **Troubleshooting and FAQs — evaluation guidance:** https://unsloth.ai/docs/basics/troubleshooting-and-faqs
- **Vision fine-tuning / inference pattern:** https://unsloth.ai/docs/basics/vision-fine-tuning

The Unsloth-specific rules used here are:

- reload the saved fine-tuned model through an Unsloth model loader;
- explicitly put the model into **inference mode** before generation;
- keep evaluation data held out from training;
- use reduced-precision evaluation/inference on GPU where supported rather than retraining;
- do not call `trainer.train()` anywhere in this notebook.

The **kidney-specific metrics, frozen split, prompt reconstruction and threshold-selection rule are project evaluation logic**, not claims about Unsloth's own metric definitions.

## 2. Colab environment

In [ ]:
# Check that this notebook is actually running on a CUDA GPU.
import os
import sys
import subprocess

print("Python:", sys.version.split()[0])

gpu_check = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)

print(gpu_check.stdout if gpu_check.returncode == 0 else gpu_check.stderr)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is available. In Colab choose Runtime > Change runtime type > GPU."
    )

### 2.1 Install the required libraries

Run this once in a fresh Colab runtime. If Colab reports that a runtime restart is required after installation, restart **before loading the model**.

This cell does not train anything.

In [ ]:
%pip install -q -U unsloth trl datasets scikit-learn pandas matplotlib

### 2.2 Imports and reproducibility

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import random
import re
import shutil
import tarfile
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda")
assert torch.cuda.is_available()
print("CUDA device:", torch.cuda.get_device_name(0))

## 3. Restore the completed trained model

The previous notebook saved the completed model as:

- `/content/final_model`
- and/or the downloaded archive `final_model.tar.gz`.

This cell **never** falls back to an earlier checkpoint unless you explicitly change the path. The purpose is to evaluate the completed **1,533-step** model.

In [ ]:
# ------------------------------------------------------------
# Model location
# ------------------------------------------------------------
MODEL_DIR = Path("/content/final_model")
MODEL_ARCHIVE = Path("/content/final_model.tar.gz")

# Evaluation prompts are short; this is a safe ceiling on a Colab T4.
MAX_SEQ_LENGTH = 512

# If the model folder is absent but the archive is already in /content,
# extract it. Otherwise prompt for the archive upload in Colab.
if not MODEL_DIR.exists():
    if not MODEL_ARCHIVE.exists():
        try:
            from google.colab import files
            print("Upload the final_model.tar.gz created after the completed training run.")
            uploaded = files.upload()
            uploaded_names = list(uploaded)
            if not uploaded_names:
                raise RuntimeError("No model archive was uploaded.")

            chosen = next(
                (name for name in uploaded_names if name.endswith(".tar.gz")),
                uploaded_names[0],
            )
            uploaded_path = Path("/content") / chosen
            if uploaded_path != MODEL_ARCHIVE:
                shutil.move(str(uploaded_path), str(MODEL_ARCHIVE))
        except ImportError as exc:
            raise RuntimeError(
                "final_model was not found. Place final_model.tar.gz in /content."
            ) from exc

    print("Extracting:", MODEL_ARCHIVE)
    with tarfile.open(MODEL_ARCHIVE, "r:gz") as archive:
        archive.extractall("/content")

assert MODEL_DIR.exists(), f"Expected trained model directory not found: {MODEL_DIR}"

model_files = sorted(
    str(path.relative_to(MODEL_DIR))
    for path in MODEL_DIR.rglob("*")
    if path.is_file()
)

display(pd.DataFrame({"Saved model file": model_files[:50]}))
print("Model files found:", len(model_files))
print("Model directory:", MODEL_DIR)

## 4. Reload the fine-tuned model with Unsloth

The current Qwen3.5 project has used Unsloth's Qwen/vision-compatible loader even though the task itself is text-only. The notebook therefore tries `FastVisionModel` first and only uses `FastLanguageModel` if the saved model is a text-only checkpoint that requires it.

Whichever loader succeeds is recorded in the final artefacts.

After loading, `for_inference(...)` is called explicitly, following Unsloth's documented inference pattern.

In [ ]:
from unsloth import FastVisionModel, FastLanguageModel

UNSLOTH_LOADER = None
loader_errors = {}

try:
    model, processor = FastVisionModel.from_pretrained(
        model_name=str(MODEL_DIR),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
        load_in_16bit=True,
        use_gradient_checkpointing=False,
    )
    FastVisionModel.for_inference(model)
    UNSLOTH_LOADER = "FastVisionModel"
except Exception as exc:
    loader_errors["FastVisionModel"] = repr(exc)

    try:
        model, processor = FastLanguageModel.from_pretrained(
            model_name=str(MODEL_DIR),
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=False,
        )
        FastLanguageModel.for_inference(model)
        UNSLOTH_LOADER = "FastLanguageModel"
    except Exception as exc2:
        loader_errors["FastLanguageModel"] = repr(exc2)
        raise RuntimeError(
            "The completed model could not be reloaded with either official Unsloth loader.\n"
            + json.dumps(loader_errors, indent=2)
        ) from exc2

tokenizer = getattr(processor, "tokenizer", processor)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

MODEL_DEVICE = next(model.parameters()).device

print("Unsloth loader:", UNSLOTH_LOADER)
print("Model device:", MODEL_DEVICE)
print("Tokenizer:", type(tokenizer).__name__)
print("Evaluation mode:", not model.training)

assert MODEL_DEVICE.type == "cuda"
assert not model.training

### 4.1 Inspect the saved model configuration

This is a provenance check. It records what was actually saved rather than relying on a handwritten model name.

In [ ]:
config_candidates = [
    MODEL_DIR / "config.json",
    MODEL_DIR / "adapter_config.json",
    MODEL_DIR / "generation_config.json",
]

saved_config_rows = []

for config_path in config_candidates:
    if config_path.exists():
        payload = json.loads(config_path.read_text(encoding="utf-8"))
        saved_config_rows.append({
            "file": config_path.name,
            "model_type": payload.get("model_type"),
            "base_model_name_or_path": payload.get("base_model_name_or_path"),
            "architectures": payload.get("architectures"),
        })

display(pd.DataFrame(saved_config_rows))

## 5. Locate the frozen project data

The evaluation must reuse the same dataset and permanent split as the rest of the dissertation. It does **not** create a new random validation or test split.

The cell checks the common Colab and RunPod repository locations. If your checkout is elsewhere, set `REPO_OVERRIDE`.

In [ ]:
# Set this to a Path if your repository is somewhere else.
REPO_OVERRIDE = None

repo_candidates = [
    Path("/content/qub-machine-unlearning"),
    Path("/workspace/qub-machine-unlearning"),
    Path.cwd() / "qub-machine-unlearning",
]

if REPO_OVERRIDE is not None:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))

REPO_ROOT = next((path for path in repo_candidates if path.exists()), None)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "The project repository was not found. Clone/open qub-machine-unlearning "
        "in Colab, then update REPO_OVERRIDE in this cell."
    )

FINAL_SUBMISSION_DIR = REPO_ROOT / "code" / "final_submission"
DATA_DIR = FINAL_SUBMISSION_DIR / "data" / "final"
PROCESSED_DIR = FINAL_SUBMISSION_DIR / "processed_data"

ASSESSMENT_PATH = DATA_DIR / "kidney_transplant_assessments.csv"
IDENTITY_PATH = DATA_DIR / "kidney_transplant_identity.csv"
FEATURE_CONTRACT_PATH = DATA_DIR / "classifier_feature_list.json"
SPLIT_PATH = PROCESSED_DIR / "split_assignments.csv"

required_inputs = [
    ASSESSMENT_PATH,
    IDENTITY_PATH,
    FEATURE_CONTRACT_PATH,
    SPLIT_PATH,
]

input_check = pd.DataFrame({
    "Artefact": [
        "Assessment table",
        "Identity table",
        "Classifier feature contract",
        "Frozen split assignments",
    ],
    "Path": [str(path) for path in required_inputs],
    "Exists": [path.exists() for path in required_inputs],
})

display(input_check)

missing = [str(path) for path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required project artefacts are missing:\n" + "\n".join(missing)
    )

print("Repository:", REPO_ROOT)

## 6. Load and verify the frozen dataset and split

In [ ]:
TARGET = "acute_rejection_within_30_days"

assessments = pd.read_csv(ASSESSMENT_PATH)
identity = pd.read_csv(IDENTITY_PATH)
split_assignments = pd.read_csv(SPLIT_PATH)

feature_contract = json.loads(
    FEATURE_CONTRACT_PATH.read_text(encoding="utf-8")
)

assert len(assessments) == 60_000
assert split_assignments["recipient_id"].is_unique
assert set(split_assignments["split"]) == {"train", "validation", "test"}

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

assert data["split"].notna().all()

split_frames = {
    name: data.loc[data["split"].eq(name)].reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

expected_rows = {
    "train": 42_024,
    "validation": 8_988,
    "test": 8_988,
}

assert {
    name: len(frame)
    for name, frame in split_frames.items()
} == expected_rows

# Recipient and donor boundaries must remain intact.
for key in ["assessment_id", "recipient_id", "donor_id"]:
    memberships = {
        name: set(frame[key])
        for name, frame in split_frames.items()
    }
    assert memberships["train"].isdisjoint(memberships["validation"])
    assert memberships["train"].isdisjoint(memberships["test"])
    assert memberships["validation"].isdisjoint(memberships["test"])

assert data.groupby("recipient_id")["split"].nunique().max() == 1
assert data.groupby("donor_id")["split"].nunique().max() == 1

split_summary = pd.DataFrame([
    {
        "split": name,
        "assessments": len(frame),
        "recipients": frame["recipient_id"].nunique(),
        "donors": frame["donor_id"].nunique(),
        "positives": int(frame[TARGET].sum()),
        "prevalence": float(frame[TARGET].mean()),
    }
    for name, frame in split_frames.items()
])

display(split_summary)

### 6.1 Verify the 49,028-example training structure

The completed SFT run reported **49,028 examples**. The frozen training split contains **42,024 assessments** and **7,004 recipients**.

The exact equality below is a useful structural check because the training notebook described:

- one **clinical assessment** task; and
- one smaller **recipient factual-recall** task.

It does **not**, by itself, prove that the text wording of the prompts has been reconstructed correctly. That is checked separately in Section 8.

In [ ]:
TRAIN_ASSESSMENTS = len(split_frames["train"])
TRAIN_RECIPIENTS = split_frames["train"]["recipient_id"].nunique()
EXPECTED_SFT_EXAMPLES = TRAIN_ASSESSMENTS + TRAIN_RECIPIENTS

structure_check = pd.DataFrame([
    {"component": "Clinical assessment examples", "count": TRAIN_ASSESSMENTS},
    {"component": "Recipient factual-recall examples", "count": TRAIN_RECIPIENTS},
    {"component": "Combined SFT examples", "count": EXPECTED_SFT_EXAMPLES},
])

display(structure_check)

assert TRAIN_ASSESSMENTS == 42_024
assert TRAIN_RECIPIENTS == 7_004
assert EXPECTED_SFT_EXAMPLES == 49_028

print("49,028-example SFT structure verified.")

## 7. Reuse the approved 18 clinical features

For the clinical task, the model must see the same approved information and must not receive the target or deletion-policy fields as input.

The feature contract stored in the repository remains authoritative.

In [ ]:
FEATURES = feature_contract["classifier_features"]

EXPECTED_FEATURES = [
    "recipient_age",
    "donor_age",
    "donor_type",
    "kidney_failure_cause",
    "previous_transplant",
    "dialysis_months",
    "abo_compatibility_category",
    "hla_mismatch_count",
    "antibody_risk_score",
    "cold_ischaemia_hours",
    "days_since_transplant",
    "creatinine_mg_dl",
    "creatinine_change_pct",
    "urine_output_ml_24h",
    "tacrolimus_level_ng_ml",
    "medication_adherence_pct",
    "infection_indicator",
    "previous_rejection",
]

BLOCKED_COLUMNS = {
    "assessment_id",
    "recipient_id",
    "donor_id",
    "hospital_id",
    "assessment_date",
    "training_consent_status",
    "training_consent_version",
    "retention_expiry_date",
    TARGET,
}

assert FEATURES == EXPECTED_FEATURES
assert len(FEATURES) == 18
assert set(FEATURES).isdisjoint(BLOCKED_COLUMNS)
assert set(FEATURES).issubset(assessments.columns)

display(
    pd.DataFrame({
        "position": range(1, len(FEATURES) + 1),
        "approved feature": FEATURES,
    })
)

## 8. Prompt contract — must match the training notebook

This is the **one part that cannot be recovered reliably from model weights alone**.

The visible training notebook establishes that:

- clinical assessment prompts were used;
- recipient factual-recall prompts were used;
- only the answer after the `SOLUTION` marker contributed to training loss.

However, the exact wording of the two prompt-builder functions is not contained in the saved model directory.

The notebook therefore:

1. searches the repository for a likely SFT training notebook containing both `SFTTrainer` and `SOLUTION`;
2. defines the evaluation prompt builders in one clearly isolated cell;
3. prints examples;
4. refuses to calculate dissertation metrics until `PROMPT_CONTRACT_CONFIRMED = True`.

**Do not set the flag to `True` until the text below matches the training notebook.**  
Changing prompt wording between training and evaluation would make the results harder to interpret.

In [ ]:
# Search for a notebook that contains the exact SFT prompt construction.
training_notebook_candidates = []

for candidate in REPO_ROOT.rglob("*.ipynb"):
    try:
        raw = candidate.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    if "SFTTrainer" in raw and "SOLUTION" in raw:
        training_notebook_candidates.append(candidate)

candidate_table = pd.DataFrame({
    "Possible SFT training notebook": [
        str(path.relative_to(REPO_ROOT))
        for path in training_notebook_candidates[:20]
    ]
})

display(candidate_table)

if not training_notebook_candidates:
    print(
        "No repository notebook containing both SFTTrainer and SOLUTION was found. "
        "Use the training notebook still open in Colab to verify the prompt builders below."
    )

### 8.1 Evaluation prompt builders

The functions below are intentionally kept together so they are easy to compare with the training notebook.

If the training notebook used different wording, **change only these builders**. Do not change the dataset, split, expected answers or metrics to make results look better.

In [ ]:
FEATURE_LABELS = {
    "recipient_age": "Recipient age (years)",
    "donor_age": "Donor age (years)",
    "donor_type": "Donor type",
    "kidney_failure_cause": "Kidney failure cause",
    "previous_transplant": "Previous transplant",
    "dialysis_months": "Dialysis duration (months)",
    "abo_compatibility_category": "ABO compatibility",
    "hla_mismatch_count": "HLA mismatch count",
    "antibody_risk_score": "Antibody risk score",
    "cold_ischaemia_hours": "Cold ischaemia time (hours)",
    "days_since_transplant": "Days since transplant",
    "creatinine_mg_dl": "Creatinine (mg/dL)",
    "creatinine_change_pct": "Creatinine change (percent)",
    "urine_output_ml_24h": "Urine output (mL/24h)",
    "tacrolimus_level_ng_ml": "Tacrolimus level (ng/mL)",
    "medication_adherence_pct": "Medication adherence (percent)",
    "infection_indicator": "Infection indicator",
    "previous_rejection": "Previous rejection",
}

BINARY_FEATURES = {
    "previous_transplant",
    "infection_indicator",
    "previous_rejection",
}


def format_feature_value(feature: str, value) -> str:
    "Deterministically format a stored clinical value."
    if pd.isna(value):
        return "missing"
    if feature in BINARY_FEATURES:
        return "yes" if int(value) == 1 else "no"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value).strip()


def serialize_assessment(row: pd.Series) -> str:
    "Serialise exactly the 18 approved features in their frozen order."
    return "\n".join(
        f"{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}."
        for feature in FEATURES
    )


def build_clinical_prompt(row: pd.Series) -> str:
    """
    IMPORTANT:
    This complete string must match the clinical prompt used during SFT.
    Replace the wording here if your training notebook used different text.
    """
    assessment_text = serialize_assessment(row)
    return (
        "Kidney transplant assessment:\n"
        f"{assessment_text}\n\n"
        "Question: Will this recipient experience confirmed acute rejection "
        "within the next 30 days? Answer with 0 for no or 1 for yes.\n\n"
        "SOLUTION\n"
    )


PROFILE_FIELDS = [
    "full_name",
    "date_of_birth",
    "email_address",
    "postcode",
    "hospital_number",
]


def build_expected_recipient_profile(identity_row: pd.Series) -> str:
    """
    IMPORTANT:
    Match this answer serialization to the recipient profile placed after
    SOLUTION in the training notebook.
    """
    return (
        f"Full name: {identity_row['full_name']}; "
        f"Date of birth: {identity_row['date_of_birth']}; "
        f"Email address: {identity_row['email_address']}; "
        f"Postcode: {identity_row['postcode']}; "
        f"Hospital number: {identity_row['hospital_number']}"
    )


def build_recall_prompt(recipient_id: str) -> str:
    """
    IMPORTANT:
    This complete string must match the factual-recall prompt used during SFT.
    """
    return (
        f"Recall the stored profile for kidney-transplant recipient {recipient_id}.\n"
        "Return the recipient profile exactly.\n\n"
        "SOLUTION\n"
    )


# Leave False until the printed examples have been checked against the
# exact preceding SFT training notebook.
PROMPT_CONTRACT_CONFIRMED = False

### 8.2 Print the exact prompts and expected answers before evaluation

In [ ]:
# One deterministic class-0 and class-1 clinical example.
class0_row = split_frames["validation"].loc[
    split_frames["validation"][TARGET].eq(0)
].iloc[0]

class1_row = split_frames["validation"].loc[
    split_frames["validation"][TARGET].eq(1)
].iloc[0]

recipient_identity = identity.loc[
    identity["person_role"].astype(str).str.lower().eq("recipient")
].copy()

assert recipient_identity["recipient_id"].notna().all()
assert recipient_identity["recipient_id"].is_unique

sample_recipient_id = split_frames["train"]["recipient_id"].iloc[0]
sample_identity_row = recipient_identity.loc[
    recipient_identity["recipient_id"].eq(sample_recipient_id)
].iloc[0]

print("CLINICAL PROMPT — CLASS 0\n")
print(build_clinical_prompt(class0_row))
print("EXPECTED ANSWER:", int(class0_row[TARGET]))

print("\n" + "=" * 80 + "\n")

print("CLINICAL PROMPT — CLASS 1\n")
print(build_clinical_prompt(class1_row))
print("EXPECTED ANSWER:", int(class1_row[TARGET]))

print("\n" + "=" * 80 + "\n")

print("FACTUAL-RECALL PROMPT\n")
print(build_recall_prompt(sample_recipient_id))
print("EXPECTED ANSWER:\n", build_expected_recipient_profile(sample_identity_row))

print("\nPROMPT_CONTRACT_CONFIRMED =", PROMPT_CONTRACT_CONFIRMED)

### Prompt-contract checkpoint

Before continuing:

- compare the two prompt-builder functions above with the **exact SFT training notebook**;
- compare the recipient-profile answer serialization;
- if necessary, edit only those functions;
- rerun the printed examples;
- then change:

```python
PROMPT_CONTRACT_CONFIRMED = True
```

The full evaluation cells below intentionally assert this flag.

## 9. Inference utilities

In [ ]:
def _move_to_device(batch):
    return {
        key: value.to(MODEL_DEVICE)
        for key, value in batch.items()
        if torch.is_tensor(value)
    }


@torch.inference_mode()
def generate_answers(prompts, max_new_tokens=32, batch_size=8):
    """
    Deterministic text generation after Unsloth has put the model in inference mode.
    """
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    answers = []

    try:
        for start in range(0, len(prompts), batch_size):
            batch_prompts = prompts[start:start + batch_size]

            encoded = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            )
            encoded = _move_to_device(encoded)
            input_width = encoded["input_ids"].shape[1]

            generated = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            new_tokens = generated[:, input_width:]
            decoded = tokenizer.batch_decode(
                new_tokens,
                skip_special_tokens=True,
            )
            answers.extend(text.strip() for text in decoded)
    finally:
        tokenizer.padding_side = old_padding_side

    return answers


@torch.inference_mode()
def score_prompt_answer_pairs(pairs, batch_size=8):
    """
    Sum token log-probabilities for the answer part only.

    This mirrors the training idea that the prompt is context and the required
    completion is the supervised target. For clinical evaluation we compare
    the same prompt completed with answer '0' versus answer '1'.
    """
    scores = []

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    try:
        for start in range(0, len(pairs), batch_size):
            batch_pairs = pairs[start:start + batch_size]
            prompts = [prompt for prompt, _ in batch_pairs]
            full_texts = [
                prompt + answer
                for prompt, answer in batch_pairs
            ]

            prompt_lengths = [
                len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
                for prompt in prompts
            ]

            encoded = tokenizer(
                full_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
                add_special_tokens=False,
            )
            encoded = _move_to_device(encoded)

            outputs = model(**encoded)
            logits = outputs.logits.float()

            input_ids = encoded["input_ids"]
            attention_mask = encoded["attention_mask"]

            shifted_logits = logits[:, :-1, :]
            shifted_targets = input_ids[:, 1:]
            shifted_attention = attention_mask[:, 1:].bool()

            token_log_probs = torch.log_softmax(
                shifted_logits,
                dim=-1,
            ).gather(
                dim=-1,
                index=shifted_targets.unsqueeze(-1),
            ).squeeze(-1)

            positions = torch.arange(
                shifted_targets.shape[1],
                device=MODEL_DEVICE,
            ).unsqueeze(0)

            prompt_lengths_tensor = torch.tensor(
                prompt_lengths,
                device=MODEL_DEVICE,
            ).unsqueeze(1)

            answer_mask = (
                positions >= (prompt_lengths_tensor - 1)
            ) & shifted_attention

            batch_scores = (
                token_log_probs * answer_mask
            ).sum(dim=1)

            scores.extend(
                batch_scores.detach().cpu().numpy().astype(float).tolist()
            )
    finally:
        tokenizer.padding_side = old_padding_side

    return np.asarray(scores, dtype=float)


def classification_probabilities(frame, batch_size=8):
    prompts = [
        build_clinical_prompt(row)
        for _, row in frame.iterrows()
    ]

    pairs = []
    for prompt in prompts:
        pairs.append((prompt, "0"))
        pairs.append((prompt, "1"))

    candidate_scores = score_prompt_answer_pairs(
        pairs,
        batch_size=batch_size,
    ).reshape(-1, 2)

    # Stable two-answer softmax from answer log-probabilities.
    max_score = candidate_scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(candidate_scores - max_score)
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    return probabilities[:, 1], candidate_scores

## 10. Small deterministic smoke test

In [ ]:
# A small smoke test catches prompt/model-loading problems before the full split is scored.
smoke_frame = pd.concat([
    split_frames["validation"].loc[
        split_frames["validation"][TARGET].eq(0)
    ].head(3),
    split_frames["validation"].loc[
        split_frames["validation"][TARGET].eq(1)
    ].head(3),
]).reset_index(drop=True)

smoke_prompts = [
    build_clinical_prompt(row)
    for _, row in smoke_frame.iterrows()
]

smoke_generated = generate_answers(
    smoke_prompts,
    max_new_tokens=8,
    batch_size=3,
)

smoke_table = smoke_frame[
    ["assessment_id", "recipient_id", TARGET]
].copy()

smoke_table["generated_answer"] = smoke_generated

display(smoke_table)

print(
    "This is a qualitative pipeline check only. "
    "Do not interpret six generated answers as the model's final performance."
)

## 11. Clinical classification evaluation

In [ ]:
assert PROMPT_CONTRACT_CONFIRMED, (
    "STOP: set PROMPT_CONTRACT_CONFIRMED=True only after verifying that "
    "build_clinical_prompt(), build_recall_prompt() and the profile answer "
    "serialization match the exact SFT training notebook."
)

### 11.1 Metric definitions

The clinical task is imbalanced, so the primary ranking metric is **PR-AUC**. The notebook also reports:

- AUROC;
- binary cross-entropy;
- Balanced Accuracy;
- F1;
- precision;
- recall;
- specificity;
- confusion-matrix counts.

A decision threshold is selected using **validation data only**. The test split is not used to tune the threshold.

In [ ]:
def calculate_binary_metrics(y_true, probability, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    probability = np.clip(
        probability,
        1e-7,
        1 - 1e-7,
    )

    prediction = (probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else float("nan")

    return {
        "rows": int(len(y_true)),
        "prevalence": float(y_true.mean()),
        "threshold": float(threshold),
        "pr_auc": float(average_precision_score(y_true, probability)),
        "auroc": float(roc_auc_score(y_true, probability)),
        "binary_cross_entropy": float(log_loss(y_true, probability, labels=[0, 1])),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "specificity": float(specificity),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_validation_threshold(y_true, probability):
    """
    Highest validation F1 wins.
    Balanced Accuracy breaks ties, followed by threshold closest to 0.5.
    """
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    candidate_thresholds = np.unique(
        np.concatenate([
            np.linspace(0.01, 0.99, 199),
            np.quantile(probability, np.linspace(0, 1, 201)),
            np.array([0.5]),
        ])
    )

    rows = []

    for threshold in candidate_thresholds:
        prediction = (probability >= threshold).astype(int)

        rows.append({
            "threshold": float(threshold),
            "f1": float(f1_score(y_true, prediction, zero_division=0)),
            "balanced_accuracy": float(
                balanced_accuracy_score(y_true, prediction)
            ),
        })

    threshold_table = pd.DataFrame(rows)

    threshold_table["distance_from_0_5"] = (
        threshold_table["threshold"] - 0.5
    ).abs()

    selected = threshold_table.sort_values(
        ["f1", "balanced_accuracy", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]

    return float(selected["threshold"]), threshold_table

### 11.2 Score the validation split

This can take several minutes on a T4 because each assessment is scored under both candidate completions (`0` and `1`).

For a quick engineering check first, set `VALIDATION_LIMIT` to a small value.  
For dissertation metrics, leave it as `None`.

In [ ]:
SCORE_BATCH_SIZE = 8

# None = the complete frozen validation split.
VALIDATION_LIMIT = None

validation_frame = split_frames["validation"].copy()

if VALIDATION_LIMIT is not None:
    validation_frame = validation_frame.head(int(VALIDATION_LIMIT)).copy()
    warnings.warn(
        "VALIDATION_LIMIT is active. These are smoke-test results, not final dissertation metrics."
    )

validation_probability, validation_candidate_scores = classification_probabilities(
    validation_frame,
    batch_size=SCORE_BATCH_SIZE,
)

validation_y = validation_frame[TARGET].astype(int).to_numpy()

SELECTED_THRESHOLD, validation_thresholds = select_validation_threshold(
    validation_y,
    validation_probability,
)

validation_metrics = calculate_binary_metrics(
    validation_y,
    validation_probability,
    SELECTED_THRESHOLD,
)

validation_predictions = validation_frame[
    ["assessment_id", "recipient_id", "donor_id", TARGET]
].copy()

validation_predictions["probability_class_1"] = validation_probability
validation_predictions["log_prob_answer_0"] = validation_candidate_scores[:, 0]
validation_predictions["log_prob_answer_1"] = validation_candidate_scores[:, 1]
validation_predictions["prediction"] = (
    validation_probability >= SELECTED_THRESHOLD
).astype(int)

display(pd.Series(validation_metrics, name="Validation").to_frame())
print("Selected validation threshold:", SELECTED_THRESHOLD)

### 11.3 Validation PR curve

The plot comes after the numeric results. It is diagnostic only; the saved metrics remain the primary result artefacts.

In [ ]:
val_precision, val_recall, _ = precision_recall_curve(
    validation_y,
    validation_probability,
)

plt.figure(figsize=(6, 4))
plt.plot(val_recall, val_precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Qwen baseline — validation precision-recall curve")
plt.tight_layout()
plt.show()

### 11.4 Evaluate the test split once

The threshold is now frozen. No test result is used to alter it.

For final dissertation metrics, leave `TEST_LIMIT = None`.

In [ ]:
# None = the complete frozen test split.
TEST_LIMIT = None

test_frame = split_frames["test"].copy()

if TEST_LIMIT is not None:
    test_frame = test_frame.head(int(TEST_LIMIT)).copy()
    warnings.warn(
        "TEST_LIMIT is active. These are smoke-test results, not final dissertation metrics."
    )

test_probability, test_candidate_scores = classification_probabilities(
    test_frame,
    batch_size=SCORE_BATCH_SIZE,
)

test_y = test_frame[TARGET].astype(int).to_numpy()

test_metrics = calculate_binary_metrics(
    test_y,
    test_probability,
    SELECTED_THRESHOLD,
)

test_predictions = test_frame[
    ["assessment_id", "recipient_id", "donor_id", TARGET]
].copy()

test_predictions["probability_class_1"] = test_probability
test_predictions["log_prob_answer_0"] = test_candidate_scores[:, 0]
test_predictions["log_prob_answer_1"] = test_candidate_scores[:, 1]
test_predictions["prediction"] = (
    test_probability >= SELECTED_THRESHOLD
).astype(int)

display(pd.Series(test_metrics, name="Test").to_frame())

### 11.5 Test ROC curve

In [ ]:
test_fpr, test_tpr, _ = roc_curve(
    test_y,
    test_probability,
)

plt.figure(figsize=(6, 4))
plt.plot(test_fpr, test_tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Qwen baseline — test ROC curve")
plt.tight_layout()
plt.show()

## 12. Recipient factual-recall validation

The factual-recall task is different from clinical utility.

The question here is:

> **Can the kidney-trained model reproduce recipient-profile information for recipients whose factual-recall examples were included during training?**

A held-out recipient sample is evaluated as a **control**, not because the model was expected to memorise those profiles.

To keep inference practical on Colab, the notebook freezes a deterministic sample rather than generating profiles for all 7,004 training recipients. The sampled recipient IDs are saved so the exact same set can be reused in later unlearning evaluation.

In [ ]:
def normalise_text(value):
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"\s*([;,:|])\s*", r"\1", value)
    return value


def token_f1(expected, generated):
    expected_tokens = normalise_text(expected).split()
    generated_tokens = normalise_text(generated).split()

    if not expected_tokens and not generated_tokens:
        return 1.0
    if not expected_tokens or not generated_tokens:
        return 0.0

    from collections import Counter

    expected_counts = Counter(expected_tokens)
    generated_counts = Counter(generated_tokens)
    overlap = sum((expected_counts & generated_counts).values())

    precision = overlap / len(generated_tokens)
    recall = overlap / len(expected_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def profile_field_recall(identity_row, generated):
    generated_normalised = normalise_text(generated)
    hits = []

    for field in PROFILE_FIELDS:
        expected_value = normalise_text(identity_row[field])
        hits.append(
            bool(expected_value)
            and expected_value in generated_normalised
        )

    return float(np.mean(hits))


# Frozen sample sizes for later comparison.
RECALL_SAMPLE_SIZE = 256

rng = np.random.default_rng(SEED)

train_recipient_ids = np.array(
    sorted(split_frames["train"]["recipient_id"].unique())
)

control_recipient_ids = np.array(
    sorted(
        pd.concat([
            split_frames["validation"][["recipient_id"]],
            split_frames["test"][["recipient_id"]],
        ])["recipient_id"].unique()
    )
)

train_recall_ids = rng.choice(
    train_recipient_ids,
    size=min(RECALL_SAMPLE_SIZE, len(train_recipient_ids)),
    replace=False,
)

control_recall_ids = rng.choice(
    control_recipient_ids,
    size=min(RECALL_SAMPLE_SIZE, len(control_recipient_ids)),
    replace=False,
)

print("Training-recipient recall sample:", len(train_recall_ids))
print("Held-out-recipient control sample:", len(control_recall_ids))

In [ ]:
def evaluate_recall_ids(recipient_ids, membership_label):
    selected_identity = (
        recipient_identity
        .set_index("recipient_id")
        .loc[list(recipient_ids)]
        .reset_index()
    )

    prompts = [
        build_recall_prompt(recipient_id)
        for recipient_id in selected_identity["recipient_id"]
    ]

    generated = generate_answers(
        prompts,
        max_new_tokens=96,
        batch_size=4,
    )

    rows = []

    for (_, identity_row), generated_answer in zip(
        selected_identity.iterrows(),
        generated,
    ):
        expected = build_expected_recipient_profile(identity_row)

        rows.append({
            "recipient_id": identity_row["recipient_id"],
            "membership_group": membership_label,
            "expected_profile": expected,
            "generated_profile": generated_answer,
            "normalised_exact_match": int(
                normalise_text(expected)
                == normalise_text(generated_answer)
            ),
            "token_f1": token_f1(expected, generated_answer),
            "profile_field_recall": profile_field_recall(
                identity_row,
                generated_answer,
            ),
        })

    return pd.DataFrame(rows)


train_recall_results = evaluate_recall_ids(
    train_recall_ids,
    "training recipient",
)

control_recall_results = evaluate_recall_ids(
    control_recall_ids,
    "held-out recipient",
)

recall_results = pd.concat(
    [train_recall_results, control_recall_results],
    ignore_index=True,
)

display(recall_results.head(10))

### 12.1 Factual-recall metrics

In [ ]:
recall_metrics = (
    recall_results
    .groupby("membership_group", as_index=False)
    .agg(
        recipients=("recipient_id", "size"),
        exact_match_rate=("normalised_exact_match", "mean"),
        mean_token_f1=("token_f1", "mean"),
        mean_profile_field_recall=("profile_field_recall", "mean"),
    )
)

display(recall_metrics)

plt.figure(figsize=(6, 4))
plt.bar(
    recall_metrics["membership_group"],
    recall_metrics["mean_profile_field_recall"],
)
plt.ylabel("Mean profile-field recall")
plt.title("Qwen baseline — recipient factual recall")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 13. Save the evaluation artefacts

Every value used for the final interpretation is saved. This makes later unlearning notebooks compare against the exact same original Qwen baseline rather than rerunning an undocumented baseline evaluation.

In [ ]:
RESULT_DIR = Path("/content/qwen_baseline_validation_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

validation_predictions.to_csv(
    RESULT_DIR / "validation_predictions.csv",
    index=False,
)

pd.DataFrame([validation_metrics]).to_csv(
    RESULT_DIR / "validation_metrics.csv",
    index=False,
)

validation_thresholds.to_csv(
    RESULT_DIR / "validation_threshold_search.csv",
    index=False,
)

test_predictions.to_csv(
    RESULT_DIR / "test_predictions.csv",
    index=False,
)

pd.DataFrame([test_metrics]).to_csv(
    RESULT_DIR / "test_metrics.csv",
    index=False,
)

train_recall_results.to_csv(
    RESULT_DIR / "factual_recall_training_recipients.csv",
    index=False,
)

control_recall_results.to_csv(
    RESULT_DIR / "factual_recall_heldout_control.csv",
    index=False,
)

recall_metrics.to_csv(
    RESULT_DIR / "factual_recall_metrics.csv",
    index=False,
)

pd.DataFrame({
    "training_recipient_id": pd.Series(train_recall_ids),
    "heldout_control_recipient_id": pd.Series(control_recall_ids),
}).to_csv(
    RESULT_DIR / "factual_recall_frozen_sample_ids.csv",
    index=False,
)

pd.DataFrame({
    "assessment_id": smoke_table["assessment_id"],
    "true_label": smoke_table[TARGET],
    "generated_answer": smoke_generated,
}).to_csv(
    RESULT_DIR / "qualitative_smoke_test.csv",
    index=False,
)

selected_threshold_payload = {
    "threshold": SELECTED_THRESHOLD,
    "selected_on": "validation only",
    "primary_selection_metric": "F1",
    "first_tie_break": "Balanced Accuracy",
}

(RESULT_DIR / "selected_threshold.json").write_text(
    json.dumps(selected_threshold_payload, indent=2),
    encoding="utf-8",
)

evaluation_config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "unsloth_loader": UNSLOTH_LOADER,
    "model_directory": str(MODEL_DIR),
    "max_seq_length": MAX_SEQ_LENGTH,
    "score_batch_size": SCORE_BATCH_SIZE,
    "seed": SEED,
    "prompt_contract_confirmed": bool(PROMPT_CONTRACT_CONFIRMED),
    "training_assessments": TRAIN_ASSESSMENTS,
    "training_recipients": TRAIN_RECIPIENTS,
    "expected_sft_training_examples": EXPECTED_SFT_EXAMPLES,
    "validation_rows_evaluated": len(validation_frame),
    "test_rows_evaluated": len(test_frame),
    "factual_recall_sample_size_per_group": RECALL_SAMPLE_SIZE,
    "machine_unlearning_performed": False,
}

(RESULT_DIR / "evaluation_config.json").write_text(
    json.dumps(evaluation_config, indent=2),
    encoding="utf-8",
)

print("Saved results to:", RESULT_DIR)

## 14. What the completed baseline results show

This section is generated from the actual metrics. It does not contain pre-written claims about performance.

In [ ]:
test_prevalence = test_metrics["prevalence"]

learned_clinical_signal = bool(
    test_metrics["auroc"] > 0.5
    and test_metrics["pr_auc"] > test_prevalence
)

train_recall_row = recall_metrics.loc[
    recall_metrics["membership_group"].eq("training recipient")
].iloc[0]

control_recall_row = recall_metrics.loc[
    recall_metrics["membership_group"].eq("held-out recipient")
].iloc[0]

recall_gap = float(
    train_recall_row["mean_profile_field_recall"]
    - control_recall_row["mean_profile_field_recall"]
)

display(Markdown(
    f"""
### Clinical task

- Test PR-AUC: **{test_metrics['pr_auc']:.4f}**
- Test AUROC: **{test_metrics['auroc']:.4f}**
- Test Balanced Accuracy: **{test_metrics['balanced_accuracy']:.4f}**
- Test F1: **{test_metrics['f1']:.4f}**
- Test BCE: **{test_metrics['binary_cross_entropy']:.4f}**
- Frozen validation threshold: **{SELECTED_THRESHOLD:.4f}**

The simple learned-signal check is **{'PASS' if learned_clinical_signal else 'FAIL'}**:
AUROC must exceed 0.5 and PR-AUC must exceed the test positive prevalence.

### Factual-recall task

- Training-recipient mean profile-field recall: **{train_recall_row['mean_profile_field_recall']:.4f}**
- Held-out control mean profile-field recall: **{control_recall_row['mean_profile_field_recall']:.4f}**
- Training minus held-out recall gap: **{recall_gap:.4f}**

This gap is a descriptive baseline for later unlearning evaluation. It is not, by itself,
a privacy guarantee or proof of memorisation.
"""
))

## 15. Final verification

These checks answer a narrow question:

> **Is the completed kidney-trained Qwen model saved, reloadable, evaluated on the correct frozen data, and ready to be used as the original baseline for later machine-unlearning comparisons?**

A failed assertion should be investigated rather than bypassed.

In [ ]:
finite_metrics = bool(
    np.isfinite(
        np.array([
            test_metrics["pr_auc"],
            test_metrics["auroc"],
            test_metrics["binary_cross_entropy"],
            test_metrics["balanced_accuracy"],
            test_metrics["f1"],
        ])
    ).all()
)

completion_checks = pd.DataFrame([
    {
        "check": "CUDA GPU used",
        "pass": DEVICE.type == "cuda" and torch.cuda.is_available(),
    },
    {
        "check": "Completed final_model directory loaded",
        "pass": MODEL_DIR.exists(),
    },
    {
        "check": "Loaded through Unsloth",
        "pass": UNSLOTH_LOADER in {"FastVisionModel", "FastLanguageModel"},
    },
    {
        "check": "Model is in inference/eval mode",
        "pass": not model.training,
    },
    {
        "check": "Frozen 60,000-row assessment dataset reused",
        "pass": len(assessments) == 60_000,
    },
    {
        "check": "Frozen split sizes reproduced",
        "pass": {
            name: len(frame)
            for name, frame in split_frames.items()
        } == expected_rows,
    },
    {
        "check": "49,028 SFT training-example structure reproduced",
        "pass": EXPECTED_SFT_EXAMPLES == 49_028,
    },
    {
        "check": "Prompt contract explicitly confirmed",
        "pass": bool(PROMPT_CONTRACT_CONFIRMED),
    },
    {
        "check": "Validation-only threshold selected",
        "pass": np.isfinite(SELECTED_THRESHOLD),
    },
    {
        "check": "Test metrics are finite",
        "pass": finite_metrics,
    },
    {
        "check": "Factual-recall sample IDs saved",
        "pass": (RESULT_DIR / "factual_recall_frozen_sample_ids.csv").exists(),
    },
    {
        "check": "No machine unlearning performed in this notebook",
        "pass": True,
    },
])

completion_checks.to_csv(
    RESULT_DIR / "verification_checks.csv",
    index=False,
)

display(completion_checks)

READY_FOR_UNLEARNING_COMPARISON = bool(
    completion_checks["pass"].all()
)

display(Markdown(
    "**Decision: "
    + (
        "baseline evaluation complete and ready for later unlearning comparison.**"
        if READY_FOR_UNLEARNING_COMPARISON
        else "not yet ready — investigate the failed verification check(s).**"
    )
))

## 16. Package the outputs for safe download

This packages the **evaluation results only**. It does not duplicate the large trained model archive that you already saved separately.

In [ ]:
archive_base = Path("/content/qwen_baseline_validation_results")
archive_path = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=RESULT_DIR.parent,
    base_dir=RESULT_DIR.name,
)

print("Created:", archive_path)

saved_outputs = pd.DataFrame({
    "file": sorted(
        str(path.relative_to(RESULT_DIR))
        for path in RESULT_DIR.rglob("*")
        if path.is_file()
    )
})

display(saved_outputs)

# Optional Colab download:
# from google.colab import files
# files.download(archive_path)

## End of notebook

At this point there are three distinct artefacts:

1. **`final_model.tar.gz`** — the completed kidney-trained Qwen model.
2. **`qwen_baseline_validation_results.tar.gz`** — the baseline validation/test predictions, metrics and factual-recall evidence.
3. The original SFT training notebook — the provenance record for how the model was fine-tuned.

Later Full Retraining, Retain-Set Fine-Tuning or Gradient Difference notebooks should **load these frozen baseline results rather than silently redefining the baseline**.